# TP3 — ACP « à la main » : géochimie des sols (mines)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/prepa/notebooks/PREPA_TP3_ACP_geochimie.ipynb)
Objectif : diagonaliser la covariance, interpréter les composantes, comparer avec la SVD.

> Données : dossier `PREPA_TP_donnees`. Règle du module : **numpy uniquement**, pas de bibliothèque d’apprentissage.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['geochimie_sols.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/prepa/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
G = pd.read_csv('geochimie_sols.csv'); print(G.head())
Z = np.log10(G.values)
Z = (Z - Z.mean(0)) / Z.std(0)
n, p = Z.shape

## 1. Covariance et valeurs propres
À COMPLÉTER : Σ = ZᵀZ / (n − 1), puis `np.linalg.eigh`.

In [ ]:
C = Z.T @ Z / (n - 1)
lam, V = np.linalg.eigh(C)
ordre = np.argsort(lam)[::-1]; lam, V = lam[ordre], V[:, ordre]
print('variance expliquée :', np.round(lam / lam.sum(), 3))
print('somme des valeurs propres =', lam.sum(), '(= p ?)')
print('V orthonormée :', np.allclose(V.T @ V, np.eye(p)))

In [ ]:
print(pd.DataFrame(V[:, :2], index=G.columns, columns=['PC1', 'PC2']).round(2))

## 2. Biplot

In [ ]:
S = Z @ V[:, :2]
plt.scatter(S[:, 0], S[:, 1], s=8, alpha=.4)
for j, c in enumerate(G.columns):
    plt.arrow(0, 0, 4 * V[j, 0], 4 * V[j, 1], color='r', head_width=.1); plt.text(4.4 * V[j, 0], 4.4 * V[j, 1], c)
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()

## 3. Même résultat par SVD

In [ ]:
U, s, Vt = np.linalg.svd(Z, full_matrices=False)
print(np.allclose(s**2 / (n - 1), lam))
err = [np.sum((Z - (U[:, :k] * s[:k]) @ Vt[:k])**2) / np.sum(Z**2) for k in range(p + 1)]
print('erreur de reconstruction :', np.round(err, 3))

## Questions
1. Quels éléments s’opposent sur PC2 ? Quelle signification géologique ?
2. Combien de composantes garder ? Justifier par deux critères.
3. Pourquoi standardiser après le log₁₀ ?